In [11]:
from pathlib import Path
import json
import random
import zipfile
import sys

MANIFEST_PATH = Path("/home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/eds_master_results_100_082_fc_d20230421_20231022.json")
EXPORT_ROOT = Path("/home/jovyan/work-easi-eds/exports/zips")
EXPORT_ROOT.mkdir(parents=True, exist_ok=True)

N_RANDOM = 5
SEED = None
CONFIRM_ZIP = False
CONFIRM_ZIP = True

# If True: only include dates where ALL components exist (FC+SR+DC4+DB8)
REQUIRE_COMPLETE = True

def parse_date_yyyymmdd_from_fc_path(p: str) -> str | None:
    name = Path(p).name
    for token in name.split("_"):
        if token.isdigit() and len(token) == 8:
            return token
    return None

def find_sr_for_date(sr_root: Path, scene: str, yyyymmdd: str) -> list[Path]:
    base = sr_root / scene / "sr"
    if not base.exists():
        return []
    return sorted(base.rglob(f"*_{yyyymmdd}_*_clr.tif"))

def find_dc4_for_date(outputs_tile_dir: Path, yyyymmdd: str, timeseries_data: str) -> list[Path]:
    dc4_tag = f"dc4{timeseries_data.lower()}"
    if not outputs_tile_dir.exists():
        return []
    return sorted([p for p in outputs_tile_dir.glob(f"*_{yyyymmdd}_{dc4_tag}*") if p.is_file()])

def find_db8_for_date(outputs_tile_dir: Path, yyyymmdd: str) -> list[Path]:
    if not outputs_tile_dir.exists():
        return []
    out = []
    imgs = sorted(outputs_tile_dir.glob(f"*_{yyyymmdd}_db8*.img"))
    for img in imgs:
        if img.exists():
            out.append(img)
            hdr = img.with_suffix(".hdr")
            aux = Path(str(img) + ".aux.xml")
            if hdr.exists(): out.append(hdr)
            if aux.exists(): out.append(aux)
    return sorted(out)

def rel_arcname(p: Path, root_parent: Path) -> str:
    try:
        return str(p.relative_to(root_parent))
    except Exception:
        return p.name

# -----------------------------
# LOAD MANIFEST
# -----------------------------
manifest = json.loads(MANIFEST_PATH.read_text())
scene = manifest.get("scene")
timeseries_data = manifest.get("timeseries_source", "fc")
fc_files = manifest["inputs"]["fc"]["matched_files"]
sr_root = Path(manifest["sr_inputs"]["sr_root"])

outputs_root = Path(manifest.get("outputs_root", f"/home/jovyan/work-easi-eds/data/compat/files/{timeseries_data}"))
outputs_tile_dir = outputs_root / scene

print("[INFO] scene:", scene)
print("[INFO] timeseries_data:", timeseries_data)
print("[INFO] fc matched_files:", len(fc_files))
print("[INFO] sr_root:", sr_root)
print("[INFO] outputs_tile_dir:", outputs_tile_dir)
sys.stdout.flush()

# -----------------------------
# BUILD (date -> fc_path)
# -----------------------------
date_to_fc = {}
for fp in fc_files:
    d = parse_date_yyyymmdd_from_fc_path(fp)
    if d:
        date_to_fc.setdefault(d, fp)

all_dates = sorted(date_to_fc.keys())
if SEED is not None:
    random.seed(SEED)

# If we require complete sets, filter candidate dates first
def is_complete_date(d: str) -> bool:
    fc_ok = Path(date_to_fc[d]).exists()
    sr_ok = len(find_sr_for_date(sr_root, scene, d)) > 0
    dc4_ok = len(find_dc4_for_date(outputs_tile_dir, d, timeseries_data)) > 0
    db8_ok = len(find_db8_for_date(outputs_tile_dir, d)) > 0
    return fc_ok and sr_ok and dc4_ok and db8_ok

candidate_dates = all_dates
if REQUIRE_COMPLETE:
    candidate_dates = [d for d in all_dates if is_complete_date(d)]
    print(f"[INFO] candidate_dates with COMPLETE bundles: {len(candidate_dates)}")
    if len(candidate_dates) < N_RANDOM:
        print("[WARN] Not enough complete dates to sample N_RANDOM; falling back to any dates.")
        candidate_dates = all_dates

chosen_dates = sorted(random.sample(candidate_dates, min(N_RANDOM, len(candidate_dates))))

# -----------------------------
# BUILD PER-DATE BUNDLES
# -----------------------------
bundles = []
for d in chosen_dates:
    fc_path = Path(date_to_fc[d])
    fc = [fc_path] if fc_path.exists() else []
    sr = find_sr_for_date(sr_root, scene, d)
    dc4 = find_dc4_for_date(outputs_tile_dir, d, timeseries_data)
    db8 = find_db8_for_date(outputs_tile_dir, d)
    bundles.append((d, fc, sr, dc4, db8))

# -----------------------------
# BUILD ZIP LIST (only from bundles)
# -----------------------------
to_zip = []
missing_notes = []
for d, fc, sr, dc4, db8 in bundles:
    if not fc:  missing_notes.append(f"[{d}] MISSING FC")
    if not sr:  missing_notes.append(f"[{d}] MISSING SR")
    if not dc4: missing_notes.append(f"[{d}] MISSING DC4{timeseries_data}")
    if not db8: missing_notes.append(f"[{d}] MISSING DB8")

    to_zip.extend(fc)
    to_zip.extend(sr)
    to_zip.extend(dc4)
    to_zip.extend(db8)

to_zip.append(MANIFEST_PATH)

unique_files = sorted({p for p in to_zip if p.exists() and p.is_file()})
zip_name = f"{scene}_{timeseries_data}_compare_random{len(chosen_dates)}_bundled.zip"
zip_path = EXPORT_ROOT / zip_name

# -----------------------------
# CLEAR MATCHING PREVIEW
# -----------------------------
print("\n" + "="*110)
print("[PREVIEW] Per-date bundles (this is what you're actually comparing)")
print("="*110)

for d, fc, sr, dc4, db8 in bundles:
    print(f"\nDATE: {d}")
    print("  FC:")
    for p in fc or ["(MISSING)"]:
        print("    -", p)
    print("  SR:")
    for p in sr or ["(MISSING)"]:
        print("    -", p)
    print("  DC4:")
    for p in dc4 or ["(MISSING)"]:
        print("    -", p)
    print("  DB8:")
    for p in db8 or ["(MISSING)"]:
        print("    -", p)

print("\n" + "-"*110)
print(f"Planned ZIP: {zip_path}")
print(f"Total files: {len(unique_files)}")
print(f"Total size (GB): {sum(p.stat().st_size for p in unique_files)/(1024**3):.3f}")
sys.stdout.flush()

if missing_notes:
    print("\n[NOTE] Missing components:")
    for m in missing_notes:
        print(" -", m)
    sys.stdout.flush()

# -----------------------------
# WRITE ZIP
# -----------------------------
if not CONFIRM_ZIP:
    print("\n[STOP] Not writing zip because CONFIRM_ZIP=False.")
    print("       If bundles look right, set CONFIRM_ZIP=True and re-run.")
else:
    if zip_path.exists():
        zip_path.unlink()

    root_parent_for_arc = outputs_tile_dir.parent if outputs_tile_dir.exists() else Path("/")

    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for p in unique_files:
            arc = rel_arcname(p, root_parent_for_arc)
            z.write(p, arcname=arc)

    print("\n[OK] Wrote zip:", zip_path)
    print("[OK] Zip size (GB):", round(zip_path.stat().st_size / (1024**3), 3))
    sys.stdout.flush()


[INFO] scene: p100r082
[INFO] timeseries_data: fc
[INFO] fc matched_files: 208
[INFO] sr_root: /home/jovyan/scratch/eds/tiles
[INFO] outputs_tile_dir: /home/jovyan/work-easi-eds/data/compat/files/fc/p100r082
[INFO] candidate_dates with COMPLETE bundles: 2
[WARN] Not enough complete dates to sample N_RANDOM; falling back to any dates.

[PREVIEW] Per-date bundles (this is what you're actually comparing)

DATE: 20180407
  FC:
    - /home/jovyan/scratch/eds/tiles/p100r082/fc/2018/201804/galsfc3_p100r082_20180407_fcm3_clr.tif
  SR:
    - /home/jovyan/scratch/eds/tiles/p100r082/sr/2018/201804/ls89sr_p100r082_20180407_nbart6m3_clr.tif
  DC4:
    - /home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_20180407_dc4fc.hdr
    - /home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_20180407_dc4fc.img
  DB8:
    - (MISSING)

DATE: 20210226
  FC:
    - /home/jovyan/scratch/eds/tiles/p100r082/fc/2021/202102/galsfc3_p100r082_20210226_fcm3_clr.tif
  SR:
    - /hom

In [12]:
from pathlib import Path
import json

# MANIFEST_PATH = Path(
#     "/home/jovyan/work-easi-eds/data/compat/files/fc/p104r074/"
#     "eds_master_results_104_074_fc_d20230503_20231026.json"
# )

with MANIFEST_PATH.open() as f:
    manifest = json.load(f)

sr = manifest["inputs"]["sr"]

sr_start_path = sr["start"]["path"]
sr_end_path   = sr["end"]["path"]

print("[SR START]")
print("  date:", sr["start"].get("effective_date"))
print("  path:", sr_start_path)
print("  exists:", Path(sr_start_path).exists())

print("\n[SR END]")
print("  date:", sr["end"].get("effective_date"))
print("  path:", sr_end_path)
print("  exists:", Path(sr_end_path).exists())


[SR START]
  date: 20230421
  path: /home/jovyan/scratch/eds/tiles/p100r082/sr/2023/202304/ls89sr_p100r082_20230421_nbart6m3_clr.tif
  exists: True

[SR END]
  date: 20231022
  path: /home/jovyan/scratch/eds/tiles/p100r082/sr/2023/202310/ls89sr_p100r082_20231022_nbart6m3_clr.tif
  exists: True


In [13]:
from pathlib import Path
import json

# MANIFEST_PATH = Path(
#     "/home/jovyan/work-easi-eds/data/compat/files/fc/p104r074/"
#     "eds_master_results_104_074_fc_d20230503_20231026.json"
# )

with MANIFEST_PATH.open() as f:
    manifest = json.load(f)

compat = manifest["outputs"]["compat"]

db8_start_path = compat.get("db8_start")
db8_end_path   = compat.get("db8_end")

print("[DB8 START]")
print("  path:", db8_start_path)
print("  exists:", Path(db8_start_path).exists() if db8_start_path else False)

print("\n[DB8 END]")
print("  path:", db8_end_path)
print("  exists:", Path(db8_end_path).exists() if db8_end_path else False)


[DB8 START]
  path: /home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_20230421_db8mz.img
  exists: True

[DB8 END]
  path: /home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_20231022_db8mz.img
  exists: True


In [14]:
from pathlib import Path
import json
import zipfile

# -----------------------------
# CONFIG
# -----------------------------
# MANIFEST_PATH = Path(
#     "/home/jovyan/work-easi-eds/data/compat/files/fc/p104r074/"
#     "eds_master_results_104_074_fc_d20230503_20231026.json"
# )

EXPORT_ZIP = Path(
    EXPORT_ROOT /
    "p104r074_sr_db8_start_end.zip"
)

EXPORT_ZIP.parent.mkdir(parents=True, exist_ok=True)

# -----------------------------
# LOAD MANIFEST
# -----------------------------
with MANIFEST_PATH.open() as f:
    manifest = json.load(f)

# SR paths (inputs.sr)
sr = manifest["inputs"]["sr"]
sr_start = Path(sr["start"]["path"])
sr_end   = Path(sr["end"]["path"])

# DB8 paths (outputs.compat)
compat = manifest["outputs"]["compat"]
db8_start = Path(compat["db8_start"])
db8_end   = Path(compat["db8_end"])

# -----------------------------
# BUILD FILE LIST
# -----------------------------
files = []

def add_with_sidecars(img_path: Path):
    files.append(img_path)
    files.append(img_path.with_suffix(".hdr"))
    files.append(Path(str(img_path) + ".aux.xml"))

# SR (tif only)
files.append(sr_start)
files.append(sr_end)

# DB8 (img + sidecars)
add_with_sidecars(db8_start)
add_with_sidecars(db8_end)

# De-duplicate + existence check
unique_files = []
for p in files:
    if p.exists() and p not in unique_files:
        unique_files.append(p)

# -----------------------------
# PREVIEW
# -----------------------------
print("\n[PREVIEW] Files to be zipped:")
print("-" * 90)

total_bytes = 0
for p in unique_files:
    size = p.stat().st_size
    total_bytes += size
    print(f"{p}  ({size/1024**2:.2f} MB)")

print("-" * 90)
print(f"Total files: {len(unique_files)}")
print(f"Total size (GB): {total_bytes/1024**3:.3f}")
print(f"Zip path: {EXPORT_ZIP}")

# -----------------------------
# ZIP
# -----------------------------
with zipfile.ZipFile(EXPORT_ZIP, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in unique_files:
        z.write(p, arcname=p.name)

print("\n[OK] Zip written:", EXPORT_ZIP)



[PREVIEW] Files to be zipped:
------------------------------------------------------------------------------------------
/home/jovyan/scratch/eds/tiles/p100r082/sr/2023/202304/ls89sr_p100r082_20230421_nbart6m3_clr.tif  (1244.63 MB)
/home/jovyan/scratch/eds/tiles/p100r082/sr/2023/202310/ls89sr_p100r082_20231022_nbart6m3_clr.tif  (1244.63 MB)
/home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_20230421_db8mz.img  (622.29 MB)
/home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_20230421_db8mz.hdr  (0.00 MB)
/home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_20230421_db8mz.img.aux.xml  (0.00 MB)
/home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_20231022_db8mz.img  (622.29 MB)
/home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_20231022_db8mz.hdr  (0.00 MB)
/home/jovyan/work-easi-eds/data/compat/files/fc/p100r082/lztmre_p100r082_20231022_db8mz.img.aux.xml  (0.00 MB)
------------------